In [23]:
from google.colab import drive
import os
import random
import numpy as np
from PIL import Image, ImageEnhance
import shutil
from collections import defaultdict

drive.mount('/content/drive')

celebrity_ids = [
    '4126', '7904', '8656', '9319', '3321', '8968', '2920', '3227', '9063', '8871',
    '7282', '7145', '3782', '8722', '3401', '164537', '4561', '2880', '3745', '3699',
    '8152', '9256', '2463', '2562', '3431', '1499', '8045', '10173', '2522', '229',
    '5239', '2425', '4394', '5260', '2837', '1158', '5698', '6098', '6568', '9151',
    '800', '818', '487', '1892', '8265', '417', '9046'
]

print(f"Working with {len(celebrity_ids)} celebrities")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working with 47 celebrities


In [26]:
base_drive_path = "/content/drive/MyDrive/celeba_full"

print("Available folders in the directory:")
try:
    available_folders = os.listdir(base_drive_path)
    celebrity_folders = [f for f in available_folders if f.startswith('Images_')]
    print(f"Found {len(celebrity_folders)} celebrity folders")
    print("First few folders:", celebrity_folders[:5])
except:
    print("Please update the base_drive_path variable with the correct path to your celebrity folders")

Available folders in the directory:
Please update the base_drive_path variable with the correct path to your celebrity folders


In [27]:
def collect_celebrity_images(base_path, celebrity_ids):
    celebrity_images = defaultdict(list)

    for celeb_id in celebrity_ids:
        folder_name = f"Images_{celeb_id}"
        folder_path = os.path.join(base_path, folder_name)

        if os.path.exists(folder_path):
            # Get all image files in the folder
            image_files = [f for f in os.listdir(folder_path)
                          if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

            # Store full paths
            full_paths = [os.path.join(folder_path, img) for img in image_files]
            celebrity_images[celeb_id] = full_paths

            print(f"Celebrity {celeb_id}: {len(full_paths)} images")
        else:
            print(f"Warning: Folder for celebrity {celeb_id} not found")

    return celebrity_images

celebrity_image_dict = collect_celebrity_images(base_drive_path, celebrity_ids)

total_images = sum(len(imgs) for imgs in celebrity_image_dict.values())
print(f"\nTotal celebrities with images: {len(celebrity_image_dict)}")
print(f"Total individual images collected: {total_images}")


Total celebrities with images: 0
Total individual images collected: 0


In [ ]:
# Comprehensive Multi-Celebrity Dataset Creator
# Strategy: Augment individuals first, then create composites, then augment composites

# ============================================================================
# CELL 1-3: [Keep the same setup, mount, and collection code from before]
# ============================================================================

# ============================================================================
# CELL 4A: Create Individual Celebrity Augmented Dataset (100 per celebrity)
# ============================================================================

def augment_single_celebrity_image(image_path, output_folder, celebrity_id, target_count=100):
    """
    Create multiple augmented versions of a single celebrity image
    """
    try:
        img = Image.open(image_path)
        base_name = f"celeb_{celebrity_id}"

        # Resize to standard size
        img = img.resize((200, 250))  # Standard celebrity size for later compositing

        augmented_images = []

        # Save original
        original_path = os.path.join(output_folder, f"{base_name}_000.jpg")
        img.save(original_path)
        augmented_images.append(original_path)

        # Create augmented versions
        for i in range(1, target_count):
            augmented = img.copy()

            # Apply random transformations
            transformations = []

            # Brightness (70% to 130%)
            if random.random() > 0.2:
                factor = random.uniform(0.7, 1.3)
                enhancer = ImageEnhance.Brightness(augmented)
                augmented = enhancer.enhance(factor)
                transformations.append(f"bright_{factor:.2f}")

            # Contrast (80% to 120%)
            if random.random() > 0.3:
                factor = random.uniform(0.8, 1.2)
                enhancer = ImageEnhance.Contrast(augmented)
                augmented = enhancer.enhance(factor)
                transformations.append(f"contrast_{factor:.2f}")

            # Color saturation (80% to 120%)
            if random.random() > 0.4:
                factor = random.uniform(0.8, 1.2)
                enhancer = ImageEnhance.Color(augmented)
                augmented = enhancer.enhance(factor)
                transformations.append(f"color_{factor:.2f}")

            # Horizontal flip
            if random.random() > 0.5:
                augmented = augmented.transpose(Image.FLIP_LEFT_RIGHT)
                transformations.append("h_flip")

            # Slight rotation (-15 to +15 degrees)
            if random.random() > 0.6:
                angle = random.uniform(-15, 15)
                augmented = augmented.rotate(angle, fillcolor='white', expand=False)
                transformations.append(f"rotate_{angle:.1f}")

            # Small scale variation (90% to 110%)
            if random.random() > 0.7:
                scale_factor = random.uniform(0.9, 1.1)
                new_size = (int(200 * scale_factor), int(250 * scale_factor))
                augmented = augmented.resize(new_size)
                # Pad or crop to maintain 200x250
                if scale_factor < 1:
                    # Pad smaller images
                    padded = Image.new('RGB', (200, 250), 'white')
                    offset = ((200 - new_size[0]) // 2, (250 - new_size[1]) // 2)
                    padded.paste(augmented, offset)
                    augmented = padded
                else:
                    # Crop larger images
                    left = (new_size[0] - 200) // 2
                    top = (new_size[1] - 250) // 2
                    augmented = augmented.crop((left, top, left + 200, top + 250))
                transformations.append(f"scale_{scale_factor:.2f}")

            # Save augmented image
            aug_filename = f"{base_name}_{i:03d}.jpg"
            aug_path = os.path.join(output_folder, aug_filename)
            augmented.save(aug_path)
            augmented_images.append(aug_path)

        return augmented_images

    except Exception as e:
        print(f"Error augmenting {image_path}: {e}")
        return []

# Create individual celebrity augmented datasets
individual_augmented_dir = "/content/individual_celebrity_augmented"
os.makedirs(individual_augmented_dir, exist_ok=True)

print("Creating 100 augmented images per celebrity...")
augmented_celebrity_dict = {}

for celeb_id in celebrity_ids[:10]:  # Start with first 10 celebrities for testing
    if celeb_id in celebrity_image_dict and celebrity_image_dict[celeb_id]:
        print(f"Processing celebrity {celeb_id}...")

        # Create folder for this celebrity
        celeb_folder = os.path.join(individual_augmented_dir, f"Celebrity_{celeb_id}")
        os.makedirs(celeb_folder, exist_ok=True)

        # Use first available image for this celebrity
        source_image = celebrity_image_dict[celeb_id][0]

        # Create 100 augmented versions
        augmented_images = augment_single_celebrity_image(
            source_image, celeb_folder, celeb_id, target_count=100
        )

        augmented_celebrity_dict[celeb_id] = augmented_images
        print(f"  ✅ Created {len(augmented_images)} images for celebrity {celeb_id}")

total_individual_images = sum(len(images) for images in augmented_celebrity_dict.values())
print(f"\n🎉 Individual augmentation complete!")
print(f"📊 Total individual celebrity images: {total_individual_images}")
print(f"📁 Celebrities processed: {len(augmented_celebrity_dict)}")

# ============================================================================
# CELL 4B: Create Multi-Celebrity Composite Images from Augmented Individuals
# ============================================================================

def create_composite_from_augmented(augmented_celebrity_dict, output_size=(640, 640), num_celebrities=3):
    """
    Create composite images using the augmented individual celebrity images
    """
    available_celebs = list(augmented_celebrity_dict.keys())

    if len(available_celebs) < num_celebrities:
        num_celebrities = len(available_celebs)

    # Randomly select celebrities
    selected_celeb_ids = random.sample(available_celebs, num_celebrities)

    # Create blank canvas
    composite = Image.new('RGB', output_size, (255, 255, 255))

    positions_info = []

    for i, celeb_id in enumerate(selected_celeb_ids):
        # Randomly select one augmented image from this celebrity
        celeb_images = augmented_celebrity_dict[celeb_id]
        if not celeb_images:
            continue

        selected_image_path = random.choice(celeb_images)

        try:
            # Load celebrity image (already 200x250)
            celeb_img = Image.open(selected_image_path)

            # Generate position with better spacing
            if num_celebrities == 2:
                positions = [(50, 150), (390, 150)]
            elif num_celebrities == 3:
                positions = [(50, 100), (390, 100), (220, 350)]
            else:  # 4 celebrities
                positions = [(50, 50), (390, 50), (50, 340), (390, 340)]

            if i < len(positions):
                x, y = positions[i]
                # Add some randomness
                x += random.randint(-30, 30)
                y += random.randint(-30, 30)

                # Ensure image fits within bounds
                x = max(0, min(x, output_size[0] - 200))
                y = max(0, min(y, output_size[1] - 250))

                # Paste celebrity image
                composite.paste(celeb_img, (x, y))

                # Store position info
                positions_info.append({
                    'celebrity_id': celeb_id,
                    'position': (x, y),
                    'size': (200, 250),
                    'bbox': (x, y, x + 200, y + 250)
                })

        except Exception as e:
            print(f"Error processing celebrity {celeb_id}: {e}")
            continue

    return composite, positions_info

# Create composite images
composite_dir = "/content/composite_images"
composite_annotations_dir = "/content/composite_annotations"
os.makedirs(composite_dir, exist_ok=True)
os.makedirs(composite_annotations_dir, exist_ok=True)

num_composites = 300  # Create 300 base composite images
print(f"Creating {num_composites} composite images from augmented individuals...")

successful_composites = 0
for i in range(num_composites):
    try:
        num_celebs = random.choice([2, 3, 4])
        composite, positions_info = create_composite_from_augmented(
            augmented_celebrity_dict,
            num_celebrities=num_celebs
        )

        if len(positions_info) > 0:
            # Save composite image
            comp_filename = f"composite_{i:04d}.jpg"
            comp_path = os.path.join(composite_dir, comp_filename)
            composite.save(comp_path)

            # Save annotations
            ann_filename = f"composite_{i:04d}.txt"
            ann_path = os.path.join(composite_annotations_dir, ann_filename)

            with open(ann_path, 'w') as f:
                f.write(f"# Composite image with {len(positions_info)} celebrities\n")
                f.write("# Format: celebrity_id, x1, y1, x2, y2\n")
                for pos_info in positions_info:
                    bbox = pos_info['bbox']
                    f.write(f"{pos_info['celebrity_id']},{bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]}\n")

            successful_composites += 1

        if (i + 1) % 50 == 0:
            print(f"Progress: {i + 1}/{num_composites} composites created")

    except Exception as e:
        print(f"Error creating composite {i}: {e}")

print(f"\n✅ Base composite creation complete!")
print(f"📊 Successfully created: {successful_composites} composite images")

# ============================================================================
# CELL 4C: Augment the Composite Images with Coordinate Transformation
# ============================================================================

def load_annotations(annotation_path):
    """Load bounding box annotations from file"""
    annotations = []
    try:
        with open(annotation_path, 'r') as f:
            lines = f.readlines()
            for line in lines:
                line = line.strip()
                if line.startswith('#') or not line:
                    continue
                parts = line.split(',')
                if len(parts) >= 5:
                    celebrity_id = parts[0]
                    x1, y1, x2, y2 = map(int, parts[1:5])
                    annotations.append({
                        'celebrity_id': celebrity_id,
                        'bbox': (x1, y1, x2, y2)
                    })
    except Exception as e:
        print(f"Error loading annotations: {e}")
    return annotations

def transform_coordinates(bbox, transformation_type, image_width=640):
    """Transform bounding box coordinates based on augmentation type"""
    x1, y1, x2, y2 = bbox

    if transformation_type == 'horizontal_flip':
        new_x1 = image_width - x2
        new_x2 = image_width - x1
        return (new_x1, y1, new_x2, y2)

    return bbox

def augment_composite_with_annotations(image_path, annotation_path, output_dir, base_name, num_augmentations=3):
    """Create augmented versions of composite images with proper coordinate transformation"""
    img = Image.open(image_path)
    annotations = load_annotations(annotation_path)

    augmented_info = []

    for aug_idx in range(num_augmentations):
        augmented = img.copy()
        transformed_annotations = []
        applied_transformations = []

        # Brightness adjustment
        if random.random() > 0.3:
            factor = random.uniform(0.8, 1.2)
            enhancer = ImageEnhance.Brightness(augmented)
            augmented = enhancer.enhance(factor)
            applied_transformations.append(f"brightness_{factor:.2f}")

        # Contrast adjustment
        if random.random() > 0.3:
            factor = random.uniform(0.9, 1.1)
            enhancer = ImageEnhance.Contrast(augmented)
            augmented = enhancer.enhance(factor)
            applied_transformations.append(f"contrast_{factor:.2f}")

        # Color saturation
        if random.random() > 0.4:
            factor = random.uniform(0.9, 1.1)
            enhancer = ImageEnhance.Color(augmented)
            augmented = enhancer.enhance(factor)
            applied_transformations.append(f"color_{factor:.2f}")

        # Horizontal flip
        flip_applied = False
        if random.random() > 0.5:
            augmented = augmented.transpose(Image.FLIP_LEFT_RIGHT)
            flip_applied = True
            applied_transformations.append("horizontal_flip")

        # Transform annotations
        for ann in annotations:
            if flip_applied:
                new_bbox = transform_coordinates(ann['bbox'], 'horizontal_flip', img.width)
            else:
                new_bbox = ann['bbox']

            transformed_annotations.append({
                'celebrity_id': ann['celebrity_id'],
                'bbox': new_bbox
            })

        # Save augmented image
        aug_filename = f"{base_name}_aug_{aug_idx}.jpg"
        aug_path = os.path.join(output_dir, aug_filename)
        augmented.save(aug_path)

        augmented_info.append({
            'image_path': aug_path,
            'annotations': transformed_annotations,
            'transformations': applied_transformations,
            'filename': aug_filename
        })

    return augmented_info

# Final augmented dataset directories
final_images_dir = "/content/final_multi_celebrity_dataset/images"
final_annotations_dir = "/content/final_multi_celebrity_dataset/annotations"
os.makedirs(final_images_dir, exist_ok=True)
os.makedirs(final_annotations_dir, exist_ok=True)

print("Augmenting composite images with coordinate transformation...")

# Copy original composites first
composite_files = [f for f in os.listdir(composite_dir) if f.endswith('.jpg')]
for comp_file in composite_files:
    # Copy original image
    src_img = os.path.join(composite_dir, comp_file)
    dst_img = os.path.join(final_images_dir, comp_file)
    shutil.copy2(src_img, dst_img)

    # Copy original annotation
    ann_file = comp_file.replace('.jpg', '.txt')
    src_ann = os.path.join(composite_annotations_dir, ann_file)
    dst_ann = os.path.join(final_annotations_dir, ann_file)
    if os.path.exists(src_ann):
        shutil.copy2(src_ann, dst_ann)

# Create augmented versions
total_final_images = len(composite_files)
augmentation_log = []

for comp_file in composite_files:
    base_name = os.path.splitext(comp_file)[0]
    comp_path = os.path.join(composite_dir, comp_file)
    ann_path = os.path.join(composite_annotations_dir, comp_file.replace('.jpg', '.txt'))

    if not os.path.exists(ann_path):
        continue

    try:
        augmented_info = augment_composite_with_annotations(
            comp_path, ann_path, final_images_dir, base_name, num_augmentations=3
        )

        # Save transformed annotations
        for aug_info in augmented_info:
            ann_filename = aug_info['filename'].replace('.jpg', '.txt')
            ann_path = os.path.join(final_annotations_dir, ann_filename)

            with open(ann_path, 'w') as f:
                f.write(f"# Augmented composite: {', '.join(aug_info['transformations'])}\n")
                f.write("# Format: celebrity_id, x1, y1, x2, y2\n")
                for ann in aug_info['annotations']:
                    bbox = ann['bbox']
                    f.write(f"{ann['celebrity_id']},{bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]}\n")

        total_final_images += len(augmented_info)
        augmentation_log.extend(augmented_info)

    except Exception as e:
        print(f"Error augmenting {comp_file}: {e}")

print(f"\n🎉 COMPLETE DATASET CREATION FINISHED!")
print(f"📊 FINAL DATASET STATISTICS:")
print(f"   • Individual celebrity images: {total_individual_images}")
print(f"   • Base composite images: {successful_composites}")
print(f"   • Final multi-celebrity dataset: {total_final_images} images")
print(f"   • Total annotations: {total_final_images}")
print(f"   • Celebrities included: {len(augmented_celebrity_dict)}")

print(f"\n📁 Final dataset location: /content/final_multi_celebrity_dataset/")
print(f"   📁 images/ - {total_final_images} composite images")
print(f"   📁 annotations/ - {total_final_images} annotation files")

# Create comprehensive summary
summary_path = "/content/final_multi_celebrity_dataset/DATASET_SUMMARY.txt"
with open(summary_path, 'w') as f:
    f.write("COMPREHENSIVE MULTI-CELEBRITY DATASET\n")
    f.write("="*50 + "\n\n")
    f.write("CREATION STRATEGY:\n")
    f.write("1. Individual Celebrity Augmentation: 100 images per celebrity\n")
    f.write("2. Composite Creation: Multi-celebrity images from augmented individuals\n")
    f.write("3. Composite Augmentation: Further augmentation with coordinate transformation\n\n")
    f.write("DATASET STATISTICS:\n")
    f.write(f"• Individual celebrity images: {total_individual_images}\n")
    f.write(f"• Base composite images: {successful_composites}\n")
    f.write(f"• Final training dataset: {total_final_images} images\n")
    f.write(f"• Celebrities included: {len(augmented_celebrity_dict)}\n")
    f.write(f"• Image dimensions: 640x640 pixels\n")
    f.write(f"• Format: JPEG with corresponding .txt annotations\n")

print(f"📄 Comprehensive summary saved: {summary_path}")

Creating 100 augmented images per celebrity...
Processing celebrity 4126...
  ✅ Created 100 images for celebrity 4126
Processing celebrity 7904...
  ✅ Created 100 images for celebrity 7904
Processing celebrity 8656...
  ✅ Created 100 images for celebrity 8656
Processing celebrity 9319...
  ✅ Created 100 images for celebrity 9319
Processing celebrity 3321...
  ✅ Created 100 images for celebrity 3321
Processing celebrity 8968...
  ✅ Created 100 images for celebrity 8968
Processing celebrity 2920...
  ✅ Created 100 images for celebrity 2920
Processing celebrity 3227...
  ✅ Created 100 images for celebrity 3227
Processing celebrity 9063...
  ✅ Created 100 images for celebrity 9063
Processing celebrity 8871...
  ✅ Created 100 images for celebrity 8871

🎉 Individual augmentation complete!
📊 Total individual celebrity images: 1000
📁 Celebrities processed: 10
Creating 300 composite images from augmented individuals...
Progress: 50/300 composites created
Progress: 100/300 composites created
Pro

In [28]:
# Install ultralytics
!pip install ultralytics

# Run the training script
# (paste the artifact code into a cell and run it)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.6 MB/s eta 0:00:00


In [ ]:
# ============================================================================
# FINAL CELL: Create ZIP and Share (No README)
# ============================================================================

import zipfile
import os
import shutil
from datetime import datetime

# Create timestamp for file naming
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

print("Creating ZIP file for sharing...")

# ============================================================================
# Create Complete Dataset ZIP
# ============================================================================

zip_path = f"/content/MultiCelebrity_Dataset_{timestamp}.zip"

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as zipf:
    # Add all images
    images_dir = "/content/final_multi_celebrity_dataset/images"
    for filename in os.listdir(images_dir):
        if filename.endswith('.jpg'):
            file_path = os.path.join(images_dir, filename)
            zipf.write(file_path, f"images/{filename}")

    # Add all annotations
    annotations_dir = "/content/final_multi_celebrity_dataset/annotations"
    for filename in os.listdir(annotations_dir):
        if filename.endswith('.txt'):
            file_path = os.path.join(annotations_dir, filename)
            zipf.write(file_path, f"annotations/{filename}")

    # Add summary file only
    summary_path = "/content/final_multi_celebrity_dataset/DATASET_SUMMARY.txt"
    if os.path.exists(summary_path):
        zipf.write(summary_path, "DATASET_SUMMARY.txt")

print(f"✅ Dataset ZIP created: {zip_path}")
print(f"📦 File size: {os.path.getsize(zip_path) / (1024*1024):.1f} MB")

# ============================================================================
# Upload to Google Drive
# ============================================================================

drive_folder = "/content/drive/MyDrive/Week2_Dataset"
os.makedirs(drive_folder, exist_ok=True)

# Copy ZIP to Drive
drive_zip_path = os.path.join(drive_folder, os.path.basename(zip_path))
shutil.copy2(zip_path, drive_zip_path)

print(f"✅ ZIP uploaded to Google Drive!")
print(f"📍 Location: {drive_zip_path}")

# ============================================================================
# Display sharing info
# ============================================================================

print(f"\n" + "="*50)
print("🎉 DATASET READY FOR SHARING!")
print("="*50)

# Count files
final_images = len([f for f in os.listdir(images_dir) if f.endswith('.jpg')])
final_annotations = len([f for f in os.listdir(annotations_dir) if f.endswith('.txt')])

print(f"\n📊 DATASET CONTENTS:")
print(f"   • Images: {final_images}")
print(f"   • Annotations: {final_annotations}")
print(f"   • File size: {os.path.getsize(zip_path) / (1024*1024):.1f} MB")

print(f"\n📤 HOW TO SHARE:")
print(f"   1. Go to Google Drive")
print(f"   2. Find: MyDrive/Week2_Dataset/{os.path.basename(zip_path)}")
print(f"   3. Right-click → Share → 'Anyone with link'")
print(f"   4. Copy and send the link")

print(f"\n💾 DIRECT DOWNLOAD:")
print(f"   Run: files.download('{zip_path}')")

# Enable direct download
try:
    from google.colab import files
    print(f"\n🔽 Click to download now:")
    # Uncomment the next line to auto-download
    # files.download(zip_path)
except:
    print(f"\nNot in Colab - file saved to: {zip_path}")

print(f"\n✨ Your dataset is ready!")

Creating ZIP file for sharing...
✅ Dataset ZIP created: /content/MultiCelebrity_Dataset_20251006_0649.zip
📦 File size: 28.3 MB
✅ ZIP uploaded to Google Drive!
📍 Location: /content/drive/MyDrive/Week2_Dataset/MultiCelebrity_Dataset_20251006_0649.zip

🎉 DATASET READY FOR SHARING!

📊 DATASET CONTENTS:
   • Images: 1200
   • Annotations: 1200
   • File size: 28.3 MB

📤 HOW TO SHARE:
   1. Go to Google Drive
   2. Find: MyDrive/Week2_Dataset/MultiCelebrity_Dataset_20251006_0649.zip
   3. Right-click → Share → 'Anyone with link'
   4. Copy and send the link

💾 DIRECT DOWNLOAD:
   Run: files.download('/content/MultiCelebrity_Dataset_20251006_0649.zip')

🔽 Click to download now:

✨ Your dataset is ready!


In [ ]:
def load_annotations(annotation_path):
    annotations = []
    try:
        with open(annotation_path, 'r') as f:
            lines = f.readlines()
            for line in lines:
                line = line.strip()
                if line.startswith('#') or not line:
                    continue
                parts = line.split(',')
                if len(parts) >= 5:
                    celebrity_id = parts[0]
                    x1, y1, x2, y2 = map(int, parts[1:5])
                    annotations.append({
                        'celebrity_id': celebrity_id,
                        'bbox': (x1, y1, x2, y2)
                    })
    except Exception as e:
        print(f"Error loading annotations: {e}")

    return annotations

def transform_coordinates(bbox, transformation_type, image_width=640):
    x1, y1, x2, y2 = bbox

    if transformation_type == 'horizontal_flip':
        # For horizontal flip: new_x = image_width - old_x
        new_x1 = image_width - x2
        new_x2 = image_width - x1
        return (new_x1, y1, new_x2, y2)

    # For brightness/contrast changes, coordinates don't change
    return bbox

def augment_image_with_annotations(image_path, annotation_path, output_folder, base_name):
    img = Image.open(image_path)
    annotations = load_annotations(annotation_path)

    augmented_info = []

    # Create augmented versions
    for aug_idx in range(2):
        augmented = img.copy()
        transformed_annotations = []
        applied_transformations = []

        # Apply brightness/contrast changes (don't affect coordinates)
        if random.random() > 0.3:
            enhancer = ImageEnhance.Brightness(augmented)
            factor = random.uniform(0.8, 1.2)
            augmented = enhancer.enhance(factor)
            applied_transformations.append(f"brightness_{factor:.2f}")

        if random.random() > 0.3:
            enhancer = ImageEnhance.Contrast(augmented)
            factor = random.uniform(0.9, 1.1)
            augmented = enhancer.enhance(factor)
            applied_transformations.append(f"contrast_{factor:.2f}")

        # Apply horizontal flip (affects coordinates)
        flip_applied = False
        if random.random() > 0.5:
            augmented = augmented.transpose(Image.FLIP_LEFT_RIGHT)
            flip_applied = True
            applied_transformations.append("horizontal_flip")

        # Transform annotations based on applied augmentations
        for ann in annotations:
            if flip_applied:
                new_bbox = transform_coordinates(ann['bbox'], 'horizontal_flip', img.width)
            else:
                new_bbox = ann['bbox']

            transformed_annotations.append({
                'celebrity_id': ann['celebrity_id'],
                'bbox': new_bbox
            })

        # Save augmented image
        aug_filename = f"{base_name}_aug_{aug_idx}.jpg"
        aug_path = os.path.join(output_folder, aug_filename)
        augmented.save(aug_path)

        augmented_info.append({
            'image_path': aug_path,
            'annotations': transformed_annotations,
            'transformations': applied_transformations
        })

    return augmented_info

# Apply augmentation with coordinate transformation
print("Applying data augmentation with coordinate transformation...")
images_dir = f"{output_dir}/images"
annotations_dir = f"{output_dir}/annotations"
augmented_dir = f"{output_dir}/augmented_images"
augmented_annotations_dir = f"{output_dir}/augmented_annotations"

os.makedirs(augmented_dir, exist_ok=True)
os.makedirs(augmented_annotations_dir, exist_ok=True)

# Copy original images and annotations
original_images = [f for f in os.listdir(images_dir) if f.endswith('.jpg')]

print("Copying original images and annotations...")
for img_file in original_images:
    # Copy image
    src_img = os.path.join(images_dir, img_file)
    dst_img = os.path.join(augmented_dir, img_file)
    shutil.copy2(src_img, dst_img)

    # Copy annotation
    ann_file = img_file.replace('.jpg', '.txt')
    src_ann = os.path.join(annotations_dir, ann_file)
    dst_ann = os.path.join(augmented_annotations_dir, ann_file)
    if os.path.exists(src_ann):
        shutil.copy2(src_ann, dst_ann)

# Create augmented versions with transformed coordinates
print("Creating augmented versions with coordinate transformation...")
augmented_count = 0
transformation_log = []

for img_file in original_images:
    base_name = os.path.splitext(img_file)[0]
    img_path = os.path.join(images_dir, img_file)
    ann_path = os.path.join(annotations_dir, img_file.replace('.jpg', '.txt'))

    if not os.path.exists(ann_path):
        print(f"Warning: No annotation file for {img_file}")
        continue

    try:
        augmented_info = augment_image_with_annotations(
            img_path, ann_path, augmented_dir, base_name
        )

        # Save transformed annotations
        for i, aug_info in enumerate(augmented_info):
            ann_filename = f"{base_name}_aug_{i}.txt"
            ann_path = os.path.join(augmented_annotations_dir, ann_filename)

            with open(ann_path, 'w') as f:
                f.write(f"# Augmented image with transformations: {', '.join(aug_info['transformations'])}\n")
                f.write("# Format: celebrity_id, x1, y1, x2, y2\n")
                for ann in aug_info['annotations']:
                    bbox = ann['bbox']
                    f.write(f"{ann['celebrity_id']},{bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]}\n")

            transformation_log.append({
                'original': img_file,
                'augmented': ann_filename.replace('.txt', '.jpg'),
                'transformations': aug_info['transformations']
            })

        augmented_count += len(augmented_info)

        if (len(transformation_log)) % 20 == 0:
            print(f"Processed {len(transformation_log)} augmented images...")

    except Exception as e:
        print(f"Error augmenting {img_file}: {e}")
        continue

# Save transformation log
log_path = os.path.join(output_dir, "augmentation_log.txt")
with open(log_path, 'w') as f:
    f.write("Data Augmentation Log\n")
    f.write("="*30 + "\n\n")
    for entry in transformation_log:
        f.write(f"Original: {entry['original']}\n")
        f.write(f"Augmented: {entry['augmented']}\n")
        f.write(f"Transformations: {', '.join(entry['transformations'])}\n")
        f.write("-" * 40 + "\n")

print(f"\n✅ Augmentation complete with coordinate transformation!")
print(f"📊 Summary:")
print(f"   • Original images: {len(original_images)}")
print(f"   • Augmented images created: {augmented_count}")
print(f"   • Total dataset size: {len(original_images) + augmented_count}")
print(f"   • All annotations properly transformed!")
print(f"📄 Transformation log saved: {log_path}")

# Verify a few transformed coordinates
print(f"\n🔍 Sample coordinate transformations:")
for i, entry in enumerate(transformation_log[:3]):
    if 'horizontal_flip' in entry['transformations']:
        print(f"   {entry['augmented']}: Coordinates flipped horizontally")
    else:
        print(f"   {entry['augmented']}: Coordinates unchanged (brightness/contrast only)")

Applying data augmentation with coordinate transformation...
Copying original images and annotations...
Creating augmented versions with coordinate transformation...
Processed 20 augmented images...
Processed 40 augmented images...
Processed 60 augmented images...
Processed 80 augmented images...
Processed 100 augmented images...
Processed 120 augmented images...
Processed 140 augmented images...
Processed 160 augmented images...
Processed 180 augmented images...
Processed 200 augmented images...
Processed 220 augmented images...
Processed 240 augmented images...
Processed 260 augmented images...
Processed 280 augmented images...
Processed 300 augmented images...
Processed 320 augmented images...
Processed 340 augmented images...
Processed 360 augmented images...
Processed 380 augmented images...
Processed 400 augmented images...

✅ Augmentation complete with coordinate transformation!
📊 Summary:
   • Original images: 200
   • Augmented images created: 400
   • Total dataset size: 600


In [ ]:
summary_path = os.path.join(output_dir, "dataset_summary.txt")
with open(summary_path, 'w') as f:
    f.write("Multi-Celebrity Dataset Summary\n")
    f.write("=" * 40 + "\n\n")
    f.write(f"Dataset created from {len(celebrity_ids)} celebrities\n")
    f.write(f"Total composite images: {successful_images}\n")
    f.write(f"Total augmented images: {len(original_images) + augmented_count}\n")
    f.write(f"Image dimensions: 640x640 pixels\n")
    f.write(f"Format: JPEG\n\n")

    f.write("Celebrity IDs used:\n")
    for celeb_id in celebrity_ids:
        if celeb_id in celebrity_image_dict:
            f.write(f"- {celeb_id} ({len(celebrity_image_dict[celeb_id])} images)\n")

# List dataset contents
print(f"\nDataset Structure:")
print(f"📁 {output_dir}/")
print(f"  📁 images/ ({len(original_images)} files)")
print(f"  📁 augmented_images/ ({len(original_images) + augmented_count} files)")
print(f"  📁 annotations/ ({len(original_images)} files)")
print(f"  📄 dataset_summary.txt")

# Create a zip file for easy sharing
import zipfile

zip_path = "/content/multi_celebrity_dataset_new.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(output_dir):
        for file in files:
            file_path = os.path.join(root, file)
            arc_name = os.path.relpath(file_path, output_dir)
            zipf.write(file_path, arc_name)

print(f"\n✅ Dataset creation complete!")
print(f"📦 Zipped dataset available at: {zip_path}")
print(f"You can download this file and share it with your team and class.")

# Show first few annotation examples
print(f"\n📋 Sample annotation format:")
sample_annotation = os.path.join(f"{output_dir}/annotations", original_images[0].replace('.jpg', '.txt'))
if os.path.exists(sample_annotation):
    with open(sample_annotation, 'r') as f:
        print(f.read())


Dataset Structure:
📁 /content/multi_celebrity_dataset_new/
  📁 images/ (200 files)
  📁 augmented_images/ (600 files)
  📁 annotations/ (200 files)
  📄 dataset_summary.txt

✅ Dataset creation complete!
📦 Zipped dataset available at: /content/multi_celebrity_dataset_new.zip
You can download this file and share it with your team and class.

📋 Sample annotation format:
# Multi-celebrity image with 3 celebrities
# Format: celebrity_id, x1, y1, x2, y2
3401,25,11,145,171
4561,340,16,460,176
6098,20,320,140,480



In [1]:
# Install ultralytics
!pip install ultralytics

# Run the training script
# (paste the artifact code into a cell and run it)

In [2]:

import os
import yaml
import shutil
from pathlib import Path
from ultralytics import YOLO
import torch

In [3]:
# ============================================================================
# STEP 1: Prepare Dataset in YOLO Format
# ============================================================================

def convert_annotations_to_yolo(dataset_dir, output_dir, celebrity_ids):
    """
    Convert bounding box annotations to YOLO format
    YOLO format: <class_id> <x_center> <y_center> <width> <height> (all normalized 0-1)
    """

    # Create celebrity ID to class index mapping
    celeb_id_to_class = {celeb_id: idx for idx, celeb_id in enumerate(celebrity_ids)}

    images_src = os.path.join(dataset_dir, "images")
    annotations_src = os.path.join(dataset_dir, "annotations")

    # Create YOLO directory structure
    yolo_train_images = os.path.join(output_dir, "train", "images")
    yolo_train_labels = os.path.join(output_dir, "train", "labels")
    yolo_val_images = os.path.join(output_dir, "val", "images")
    yolo_val_labels = os.path.join(output_dir, "val", "labels")

    for path in [yolo_train_images, yolo_train_labels, yolo_val_images, yolo_val_labels]:
        os.makedirs(path, exist_ok=True)

    # Get all annotation files
    annotation_files = sorted([f for f in os.listdir(annotations_src) if f.endswith('.txt')])

    # Split into train (80%) and validation (20%)
    split_idx = int(len(annotation_files) * 0.8)
    train_files = annotation_files[:split_idx]
    val_files = annotation_files[split_idx:]

    print(f"Converting annotations to YOLO format...")
    print(f"  Train: {len(train_files)} images")
    print(f"  Val: {len(val_files)} images")

    def process_split(files, images_dir, labels_dir):
        converted_count = 0
        for ann_file in files:
            img_file = ann_file.replace('.txt', '.jpg')
            img_src = os.path.join(images_src, img_file)
            ann_src = os.path.join(annotations_src, ann_file)

            if not os.path.exists(img_src):
                continue

            # Copy image
            shutil.copy2(img_src, os.path.join(images_dir, img_file))

            # Convert annotation
            yolo_annotations = []
            with open(ann_src, 'r') as f:
                for line in f:
                    line = line.strip()
                    if line.startswith('#') or not line:
                        continue

                    parts = line.split(',')
                    if len(parts) >= 5:
                        celeb_id = parts[0]
                        x1, y1, x2, y2 = map(int, parts[1:5])

                        # Skip if celebrity not in mapping
                        if celeb_id not in celeb_id_to_class:
                            continue

                        # Convert to YOLO format (normalized coordinates)
                        img_width = 640  # Your image width
                        img_height = 640  # Your image height

                        x_center = ((x1 + x2) / 2) / img_width
                        y_center = ((y1 + y2) / 2) / img_height
                        width = (x2 - x1) / img_width
                        height = (y2 - y1) / img_height

                        class_id = celeb_id_to_class[celeb_id]

                        yolo_annotations.append(f"{class_id} {x_center} {y_center} {width} {height}")

            # Write YOLO annotation file
            if yolo_annotations:
                yolo_ann_path = os.path.join(labels_dir, ann_file)
                with open(yolo_ann_path, 'w') as f:
                    f.write('\n'.join(yolo_annotations))
                converted_count += 1

        return converted_count

    train_count = process_split(train_files, yolo_train_images, yolo_train_labels)
    val_count = process_split(val_files, yolo_val_images, yolo_val_labels)

    print(f"✅ Conversion complete!")
    print(f"  Train: {train_count} images")
    print(f"  Val: {val_count} images")

    return celeb_id_to_class

In [4]:

# ============================================================================
# STEP 2: Create YOLO Configuration File
# ============================================================================

def create_yolo_config(output_dir, celebrity_ids, celeb_id_to_class):
    """Create data.yaml configuration file for YOLO training"""

    config = {
        'path': output_dir,  # Dataset root directory
        'train': 'train/images',  # Train images
        'val': 'val/images',  # Validation images
        'nc': len(celebrity_ids),  # Number of classes
        'names': {idx: celeb_id for celeb_id, idx in celeb_id_to_class.items()}
    }

    config_path = os.path.join(output_dir, 'data.yaml')
    with open(config_path, 'w') as f:
        yaml.dump(config, f, sort_keys=False)

    print(f"📄 Config file created: {config_path}")
    return config_path

In [5]:
# ============================================================================
# STEP 3: Train YOLOv8 Model
# ============================================================================

def train_yolo_model(config_path, epochs=100, batch_size=16, img_size=640):
    """
    Train YOLOv8 model for celebrity detection
    """

    print(f"\n{'='*50}")
    print(f"🚀 Starting YOLOv8 Training")
    print(f"{'='*50}\n")

    # Check if GPU is available
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Training device: {device}")

    # Load YOLOv8 model (using nano version for faster training)
    # Options: yolov8n.pt (nano), yolov8s.pt (small), yolov8m.pt (medium)
    model = YOLO('yolov8n.pt')

    # Train the model
    results = model.train(
        data=config_path,
        epochs=epochs,
        imgsz=img_size,
        batch=batch_size,
        device=device,
        patience=20,  # Early stopping patience
        save=True,
        plots=True,  # Save training plots
        name='celebrity_detection',
        exist_ok=True,
        verbose=True
    )

    print(f"\n✅ Training complete!")
    print(f"📊 Results saved to: runs/detect/celebrity_detection")

    return model, results



In [6]:
# ============================================================================
# STEP 4: Evaluate Model
# ============================================================================

def evaluate_model(model, config_path):
    """Evaluate the trained model on validation set"""

    print(f"\n{'='*50}")
    print(f"📊 Evaluating Model")
    print(f"{'='*50}\n")

    metrics = model.val(data=config_path)

    print(f"\nValidation Results:")
    print(f"  mAP50: {metrics.box.map50:.4f}")
    print(f"  mAP50-95: {metrics.box.map:.4f}")
    print(f"  Precision: {metrics.box.mp:.4f}")
    print(f"  Recall: {metrics.box.mr:.4f}")

    return metrics

# ============================================================================
# STEP 5: Test Inference
# ============================================================================

def test_inference(model, test_image_path, celebrity_ids):
    """Run inference on a test image"""

    print(f"\n{'='*50}")
    print(f"🔍 Testing Inference")
    print(f"{'='*50}\n")

    results = model.predict(
        source=test_image_path,
        save=True,
        save_txt=True,
        conf=0.25,  # Confidence threshold
        iou=0.45,   # IoU threshold for NMS
        show_labels=True,
        show_conf=True
    )

    # Parse results
    for result in results:
        boxes = result.boxes
        for box in boxes:
            class_id = int(box.cls[0])
            confidence = float(box.conf[0])
            bbox = box.xyxy[0].tolist()  # [x1, y1, x2, y2]

            celeb_id = celebrity_ids[class_id]
            print(f"  Detected: Celebrity {celeb_id}")
            print(f"    Confidence: {confidence:.2f}")
            print(f"    BBox: {bbox}")

    print(f"\n✅ Predictions saved to: runs/detect/predict")

    return results




In [7]:



# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":

    # Configuration
    DATASET_DIR = "/content/drive/MyDrive/deeplearningproject"
    OUTPUT_DIR = "/content/yolo_celebrity_dataset"

    # Celebrity IDs from your notebook
    celebrity_ids = [
        '4126', '7904', '8656', '9319', '3321', '8968', '2920', '3227', '9063', '8871'
    ]

    # Step 1: Convert dataset to YOLO format
    print("Step 1: Converting dataset to YOLO format...")
    celeb_id_to_class = convert_annotations_to_yolo(
        DATASET_DIR,
        OUTPUT_DIR,
        celebrity_ids
    )

    # Step 2: Create YOLO config
    print("\nStep 2: Creating YOLO configuration...")
    config_path = create_yolo_config(
        OUTPUT_DIR,
        celebrity_ids,
        celeb_id_to_class
    )

    # Step 3: Train model
    print("\nStep 3: Training YOLOv8 model...")
    model, results = train_yolo_model(
        config_path,
        epochs=10,  # Adjust based on your needs
        batch_size=16,  # Adjust based on your GPU memory
        img_size=640
    )

    # Step 4: Evaluate model
    print("\nStep 4: Evaluating model...")
    metrics = evaluate_model(model, config_path)

    # Step 5: Test on sample image (optional)
    test_image = os.path.join(OUTPUT_DIR, "val/images")
    test_images = [f for f in os.listdir(test_image) if f.endswith('.jpg')]
    if test_images:
        sample_image = os.path.join(test_image, test_images[0])
        print(f"\nStep 5: Testing inference on: {sample_image}")
        test_inference(model, sample_image, celebrity_ids)

    # Save final model
    final_model_path = "/content/celebrity_detector_final.pt"
    shutil.copy2("runs/detect/celebrity_detection/weights/best.pt", final_model_path)
    print(f"\n🎉 Training pipeline complete!")
    print(f"📦 Best model saved to: {final_model_path}")
    print(f"\nTo use your model:")
    print(f"  from ultralytics import YOLO")
    print(f"  model = YOLO('{final_model_path}')")
    print(f"  results = model.predict('your_image.jpg')")

Step 1: Converting dataset to YOLO format...
Converting annotations to YOLO format...
  Train: 960 images
  Val: 240 images
✅ Conversion complete!
  Train: 960 images
  Val: 240 images

Step 2: Creating YOLO configuration...
📄 Config file created: /content/yolo_celebrity_dataset/data.yaml

Step 3: Training YOLOv8 model...

🚀 Starting YOLOv8 Training

Training device: cpu
Ultralytics 8.3.205 🚀 Python-3.12.11 torch-2.8.0+cu126 CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_celebrity_dataset/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=Fals

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
